# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\nPublished: {getattr(metadata, 'datePublished', 'N/A')}\nVersion: {getattr(metadata, 'version', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Get a list of record set IDs
record_sets = [rs['@id'] for rs in metadata.record_sets]
print(f"Record sets found in this dataset:\n{record_sets}\n")

# For each record set, print field info
for rs in metadata.record_sets:
    print(f"--- Record Set: {rs['name']} (ID: {rs['@id']}) ---")
    if 'fields' in rs:
        for field in rs['fields']:
            print(f"  Field: {field['name']}, Field @id: {field['@id']}, DataType: {field.get('dataType', 'N/A')}")
    elif 'columns' in rs:
        for col in rs['columns']:
            print(f"  Column: {col['name']}, Column @id: {col['@id']}, DataType: {col.get('dataType', 'N/A')}")
    else:
        print("  No fields or columns found.")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set

# For this dataset we pick the first record set
dataframes = {}
if len(record_sets) == 0:
    print("No record sets found in this dataset.")
else:
    # It's common to use the first record set for illustration
    target_record_set_id = record_sets[0]
    print(f"Working with Record Set ID: {target_record_set_id}")
    records = list(dataset.records(record_set=target_record_set_id))
    df = pd.DataFrame(records)
    dataframes[target_record_set_id] = df
    print(f"Available DataFrame columns for record set {target_record_set_id}:")
    print(df.columns.tolist())
    df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

if len(record_sets) == 0:
    print("No record sets found for EDA.")
else:
    df = dataframes[target_record_set_id]

    # Try to infer a numeric field from the columns
    numeric_field_id = None
    preferred_numeric_names = ['Abundance', 'abundance', 'Molecular Weight', 'MW', 'Coverage', 'PeptideCount', 'sequence_coverage', 'peptide_count', 'total_intensity', 'Score', 'score']
    for col in df.columns:
        if any(keyword.lower() in col.lower() for keyword in preferred_numeric_names):
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break

    # Otherwise, choose the first numeric column
    if numeric_field_id is None:
        for col in df.select_dtypes(include=[np.number]).columns:
            numeric_field_id = col
            break

    if numeric_field_id is None:
        print("No numeric fields found to analyze.")
    else:
        print(f"Using numeric field: {numeric_field_id}")
        threshold = df[numeric_field_id].mean()  # Use mean as example threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (mean value):\n")
        print(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by a categorical field, guess one
        group_field = None
        preferred_group_names = ['description', 'Description', 'accession', 'Accession', 'experiment', 'group', 'sample_label', 'Sample', 'Set']
        for col in filtered_df.columns:
            if col != numeric_field_id and any(kw.lower() in col.lower() for kw in preferred_group_names):
                if not pd.api.types.is_numeric_dtype(filtered_df[col]):
                    group_field = col
                    break
        if group_field is None:
            obj_cols = filtered_df.select_dtypes(include=['object']).columns
            group_field = obj_cols[0] if len(obj_cols) else None

        if group_field is not None and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index().sort_values(by=numeric_field_id, ascending=False)
            print(f"\nGrouped (mean) {numeric_field_id} by {group_field}:")
            print(grouped_df.head())
        else:
            print("No appropriate categorical field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(record_sets) == 0 or numeric_field_id is None:
    print("No numeric field and/or data available for visualization.")
else:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If group_field is available, show bar plot of means
    if 'group_field' in locals() and group_field is not None and group_field in df.columns:
        plt.figure(figsize=(10,4))
        tmp = df.groupby(group_field)[numeric_field_id].mean().sort_values(ascending=False).head(20) # Top 20 groups
        sns.barplot(x=tmp.values, y=tmp.index, orient='h')
        plt.title(f"Top 20 {group_field} by mean {numeric_field_id}")
        plt.xlabel(f"Mean {numeric_field_id}")
        plt.ylabel(group_field)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded the FAIR^2 dataset describing protein abundance and modifications as measured by mass spectrometry in extracellular vesicles from human mast cells. We explored the available record sets, examined all fields and their corresponding `@id`s, and loaded the data into pandas DataFrames for analysis.

We performed basic EDA to filter and normalize a chosen numeric field, grouped data by a categorical variable where available, and visualized the main distributions. This workflow can be extended to more advanced analyses relevant to proteomics or experimental biology.

For your own analysis, use the exact field and column `@id`s (from Section 2) for future automation or to script custom data extraction from this or other Croissant-compliant datasets.